# Bake-off: choosing an anomaly detector

sktime ships **22 detectors**. Which one should an agent use on your chiller?

This notebook does what an agent would do: discover the candidates, register them as catalog
cards, run each through the **same** `run_recipe` tool, score them against known ground truth,
and record the decision back in the catalog.

The punchline arrives early: **three detectors achieve perfect recall, and only one of them is
worth using.** Selecting on 'did it find the anomalies?' alone will pick a detector that flags
half your data.

No CouchDB needed — in-memory store. Set `TSFM_STORE=couch` for the real thing.

In [ ]:
import json, os, sys

REPO = os.path.abspath(os.environ.get('AOB_REPO', '.'))
SRC  = os.path.join(REPO, 'src')
sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['TSFM_STORE'] = 'memory'

from mcphub import ToolUniverse
SERVER_CMD = os.environ.get('SERVER_CMD', f'{sys.executable} -m servers.tsfm.main').split()
tu = ToolUniverse(servers={'tsfm': SERVER_CMD})
print(f'{tu.load_tools(servers=["tsfm"])} tools discovered')

def run(name, args=None):
    r = tu.run({'name': f'tsfm.{name}', 'arguments': args or {}})
    return r['result'] if isinstance(r, dict) and set(r) == {'result'} else r

## 1. Data with known ground truth

We inject spikes at **t = 70, 150, 240**. Knowing the answer is what makes a bake-off possible —
in production you would use a labelled incident window instead.

In [ ]:
import numpy as np, pandas as pd
from servers.tsfm.io import refs

N = 300; t = np.arange(N)
signal = 10 + np.sin(t / 24 * 2 * np.pi) * 3 + 0.01 * t + np.random.RandomState(0).normal(0, .2, N)
TRUTH = [70, 150, 240]
for i in TRUTH: signal[i] += 8            # the anomalies we plant

CHILLER = refs.materialize_iot(signal, asset_id='chiller6')
print('data      :', CHILLER)
print('ground truth spikes at:', TRUTH)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(signal, lw=.9, label='chiller 6')
ax.scatter(TRUTH, signal[TRUTH], c='crimson', zorder=5, s=60, label='injected anomaly')
ax.set_title('Chiller 6: daily cycle + drift, with 3 injected spikes')
ax.legend(); plt.tight_layout(); plt.show()

## 2. What detectors exist?

sktime's registry is the menu. Note the mix: some are genuinely anomaly detectors, some are
**change-point** or **segmentation** algorithms, and some are deliberate dummies. They all
satisfy the same interface — which is exactly why you have to measure rather than assume.

In [ ]:
from sktime.registry import all_estimators
det = all_estimators(estimator_types='detector', return_names=True)
print(f'{len(det)} detectors in sktime\n')
print(', '.join(n for n, _ in det))

## 3. Register the candidates as catalog cards

Each becomes a **card**: a pointer that says how to construct the detector. This is the same
`register_model` an agent uses — the bake-off runs entirely through the catalog.

I include two dummies on purpose. A baseline that *cannot* be good is the cheapest way to catch
a scoring mistake.

In [ ]:
CANDIDATES = {
    'sublof':    ('sktime.detection.lof.SubLOF',
                  {'window_size': 24, 'n_neighbors': 5, 'novelty': True},
                  'Sub-sequence Local Outlier Factor: density-based, windowed.'),
    'hampel':    ('sktime.detection.hampel.HampelDetector', {},
                  'Hampel filter: rolling median + MAD; classic spike detector.'),
    'pelt':      ('sktime.detection.pelt.PELT', {},
                  'PELT: a CHANGE-POINT detector - included to show the wrong tool.'),
    'dummy_regular': ('sktime.detection.dummy._dummy_regular_an.DummyRegularAnomalies', {},
                  'Dummy: flags points at a regular interval. A deliberate baseline.'),
    'zero':      ('sktime.detection.dummy._zero_an.ZeroAnomalies', {},
                  'Dummy: flags nothing. The null baseline.'),
}

for mid, (cls, params, desc) in CANDIDATES.items():
    r = run('register_model', {'model': {
        'model_id': mid, 'description': desc, 'task_ids': ['tsfm_anomaly_detection'],
        'sktime_class': cls, 'params': params, 'domain': 'energy'}})
    print(f"  {mid:14s} {r.get('status', r.get('error'))}")

found = run('find_models', {'task_id': 'tsfm_anomaly_detection', 'domain': 'energy'})
print('\ncatalog now offers:', [m['model_id'] for m in found['models']])

### Preflight them all

`resolve_model` confirms each card can actually be constructed — **before** we spend time
running anything. Cheap insurance against a typo in a class path.

In [ ]:
for mid in CANDIDATES:
    r = run('resolve_model', {'model_id': mid})
    print(f"  {mid:14s} resolvable={r.get('resolvable')}  regime={r.get('training_regime')}")

## 4. How we score

A flag within ±2 samples of a planted spike counts as a hit.

* **recall** — of the 3 spikes, how many were found?
* **precision** — of the flags raised, how many were real?

Precision is what stops a detector from 'winning' by flagging everything. In an ops setting it
*is* the alert-fatigue metric.

In [ ]:
TOL = 2

def score(labels):
    flagged = [i for i, v in enumerate(labels) if v]
    tp = sum(1 for s in TRUTH if any(abs(f - s) <= TOL for f in flagged))   # distinct spikes hit
    fp = sum(1 for f in flagged if all(abs(f - s) > TOL for s in TRUTH))    # flags on clean data
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / len(TRUTH)
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(flagged=len(flagged), tp=tp, fp=fp,
                precision=round(prec, 3), recall=round(rec, 3), f1=round(f1, 3))

## 5. The bake-off

Every candidate goes through the **same** `run_recipe` call. Only the `model_id` changes —
the recipe is the agent's decision surface, and this is what varying one decision looks like.

In [ ]:
import time
rows = []
for mid in CANDIDATES:
    t0 = time.time()
    r = run('run_recipe', {
        'dataset_path': CHILLER, 'timestamp_column': 'timestamp', 'target_columns': ['value'],
        'asset_id': 'chiller6',
        'recipe': {'task': 'tsfm_anomaly_detection', 'estimator': {'model_id': mid}},
    })
    if 'error' in r:
        print(f'  {mid:14s} ERROR: {r["error"][:60]}'); continue
    labels = json.load(open(refs._path(r['results_file'])))['anomaly_label']
    rows.append({'detector': mid, **score(labels), 'secs': round(time.time() - t0, 1),
                 'run_id': r['run_id'], '_labels': labels})

board = pd.DataFrame(rows).drop(columns=['_labels']).sort_values('f1', ascending=False)
board.reset_index(drop=True)

## 6. Read the table carefully

**Three detectors have recall = 1.0.** If you selected on recall, you might ship `dummy_regular`,
which flags **150 of 300 points** — every spike found, and completely useless.

* `sublof` — 3 flags, 3 hits, **0 false positives**. Precision 1.0.
* `hampel` — precise but found only 1 of 3.
* `pelt` — perfect recall, precision 0.12. It is a *change-point* detector: it fires on the level
  shift around each spike, not the spike. Wrong tool, plausible-looking output.
* `dummy_regular` — perfect recall by brute force. Precision 0.02.
* `zero` — flags nothing. Scores 0 everywhere, exactly as it should.

The dummies are doing their job: they prove the metric is measuring something real.

In [ ]:
fig, axes = plt.subplots(len(rows), 1, figsize=(11, 1.5 * len(rows)), sharex=True)
for ax, row in zip(axes, rows):
    ax.plot(signal, lw=.7, color='#888')
    flags = [i for i, v in enumerate(row['_labels']) if v]
    ax.scatter(flags, signal[flags], c='tab:orange', s=22, label=f"flagged ({len(flags)})")
    ax.scatter(TRUTH, signal[TRUTH], facecolors='none', edgecolors='crimson', s=110, lw=1.6,
               label='truth')
    ax.set_ylabel(row['detector'], rotation=0, ha='right', va='center', fontsize=9)
    ax.legend(loc='upper right', fontsize=7)
axes[0].set_title('What each detector flagged (orange) vs the truth (red circles)')
plt.tight_layout(); plt.show()

## 7. Make the decision, and record it

This is the part that closes the loop. The bake-off produced evidence; now we write the
conclusion back into the catalog so the next agent inherits it instead of re-deriving it.

`update_model` promotes the winner. `deprecate_model` retires the ones that lost — a soft
delete, so they stay auditable but drop out of `find_models`.

In [ ]:
winner = board.iloc[0]['detector']
print(f'winner: {winner}  (F1={board.iloc[0]["f1"]}, precision={board.iloc[0]["precision"]})\n')

run('update_model', {'model_id': winner, 'fields': {
    'tags': ['bake-off-winner', 'chiller'],
    'description': (CANDIDATES[winner][2] +
                    f' Selected for chiller 6: F1={board.iloc[0]["f1"]}, '
                    f'precision={board.iloc[0]["precision"]} against 3 known spikes.')}})

for mid in board.iloc[1:]['detector']:
    r = run('deprecate_model', {'model_id': mid,
            'reason': f'lost the chiller-6 bake-off (F1 {board.set_index("detector").loc[mid, "f1"]})'})
    print(f"  retired {mid:14s} -> {r['status']}")

left = run('find_models', {'task_id': 'tsfm_anomaly_detection', 'domain': 'energy'})
print('\nfind_models now returns only:', [m['model_id'] for m in left['models']])
print('^ the next agent gets the winner, without re-running the bake-off')

### The losers are retired, not deleted

Deprecation is a soft delete: the cards remain for audit, carrying *why* they lost.

In [ ]:
d = run('describe_models', {'model_ids': list(CANDIDATES)})
for m in d['models']:
    print(f"  {m['model_id']:14s} {str(m.get('tags')):32s}")
print('\nunknown:', d['unknown'])

# each retired card carries WHY it lost - fetch them by id, not by position
print('\nretirement reasons:')
for m in run('list_models', {'status': 'deprecated'})['models']:
    print(f"  {m['model_id']:14s} {m.get('deprecation_reason', '-')}")

## 8. Every run is in the ledger

The bake-off is reproducible: each candidate's run was recorded, so the evidence behind the
decision is inspectable later.

In [ ]:
runs = run('list_runs', {'asset_id': 'chiller6'})
print(f"{len(runs['runs'])} runs recorded\n")
for r in runs['runs']:
    print(f"  {r['run_id']:18s} regime={r.get('training_regime')}")

one = run('get_run', {'run_id': rows[0]['run_id']})
print(f"\nget_run({rows[0]['run_id']}) -> recipe that produced it:")
print(' ', json.dumps(one.get('recipe'), default=str)[:110])

## What this showed

```
find_models    ->  what can detect anomalies?
resolve_model  ->  can each candidate actually load?          (before spending time)
run_recipe     ->  same tool, same call, only model_id varies
score          ->  precision AND recall against known truth
update_model   ->  promote the winner, with the evidence
deprecate_model->  retire the losers, with the reason
```

Two things worth taking away.

**The catalog is how a decision survives.** The bake-off is the easy part; writing the outcome
back is what stops the next agent redoing it.

**The server never chose.** It ran exactly what each recipe said and recorded the results. Every
judgement — which candidates, which metric, which tolerance, who won — was the agent's. That is
the design: evidence from the server, decisions from the agent.

In [ ]:
tu.close()
print('closed')